```mermaid
graph TD
    A[用户请求] --> B{JobMatchingAgent}
    B --> C[硬性条件过滤]
    B --> D[向量相似度搜索]
    C --> E[合并结果]
    D --> E
    E --> F[精排]
    F --> G[返回结果]
    
    subgraph Zilliz Cloud
        H[(向量索引)]
        I[(标量数据)]
    end
    C --> I
    D --> H
```

In [1]:
import os

from dotenv import load_dotenv
from scrapy.utils.project import get_project_settings

from app.config import get_project_root

# load .env
load_dotenv(os.path.join(get_project_root(), ".env"))

# logger = logging.getLogger(__name__)
settings = get_project_settings()

In [2]:
from contextlib import contextmanager
from sqlalchemy.orm import sessionmaker

@contextmanager
def session_scope(sessionmaker):
    """Provide a transactional scope around a series of operations."""
    session = sessionmaker()
    try:
        yield session
        session.commit()
    except:
        session.rollback()
        raise
    finally:
        session.close()

In [3]:
from app.services.storage.engine import engine

In [4]:
from app.models.constant import JobSource
from app.models.job import JobItem, JobSource
from sqlalchemy import insert, select, and_

with session_scope(sessionmaker(bind=engine)) as session:
    jb_item = session.scalars(
        select(JobItem).where(JobItem.source==JobSource.ZHILIAN)
    ).first()
    session.expunge(jb_item)

2025-08-05 14:11:17,781 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-08-05 14:11:17,785 INFO sqlalchemy.engine.Engine SELECT job_item_db.id, job_item_db.source, job_item_db.url, job_item_db.job_title, job_item_db.update_time, job_item_db.location, job_item_db.recruitment_type, job_item_db.description, job_item_db.company_name 
FROM job_item_db 
WHERE job_item_db.source = ?
2025-08-05 14:11:17,786 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ('ZHILIAN',)


2025-08-05 14:11:17,902 INFO sqlalchemy.engine.Engine COMMIT


In [5]:
jb_item.job_title, jb_item.description, jb_item.url

('兼职/时间自由/周结',
 '工作内容:语音直播，声音好听，不需要露脸，厅播，纯绿色直播(不涉及任何违法违 规的 内容)岗位介绍:不露脸的聊天主播，会聊天即可。接受新人小白，免费培训，有很大晋升空间。 岗位要求:1.年龄:33岁以下 2.普通话标准，善于沟通 3.快手、抖音APP进行实名认证 4.在家即可 不需要来公司 5.每天直播四个小时，分两场(全天24小时自选) 注:1.免费培训',
 'https://www.zhaopin.com/jobdetail/CCL1498280210J40829129608.htm')

In [17]:
len(str(jb_item.id))

36

In [6]:
import json

jb_item_text = json.dumps({"岗位名称":jb_item.job_title, "工作描述":jb_item.description}, ensure_ascii=False, )

In [7]:
print(jb_item)

{"岗位名称": "兼职/时间自由/周结", "工作描述": "工作内容:语音直播，声音好听，不需要露脸，厅播，纯绿色直播(不涉及任何违法违 规的 内容)岗位介绍:不露脸的聊天主播，会聊天即可。接受新人小白，免费培训，有很大晋升空间。 岗位要求:1.年龄:33岁以下 2.普通话标准，善于沟通 3.快手、抖音APP进行实名认证 4.在家即可 不需要来公司 5.每天直播四个小时，分两场(全天24小时自选) 注:1.免费培训"}


In [11]:
str(jb_item)

'{"岗位名称": "兼职/时间自由/周结", "工作描述": "工作内容:语音直播，声音好听，不需要露脸，厅播，纯绿色直播(不涉及任何违法违 规的 内容)岗位介绍:不露脸的聊天主播，会聊天即可。接受新人小白，免费培训，有很大晋升空间。 岗位要求:1.年龄:33岁以下 2.普通话标准，善于沟通 3.快手、抖音APP进行实名认证 4.在家即可 不需要来公司 5.每天直播四个小时，分两场(全天24小时自选) 注:1.免费培训"}'

In [10]:
"try: {jb_item}".format(jb_item = jb_item)

'try: {"岗位名称": "兼职/时间自由/周结", "工作描述": "工作内容:语音直播，声音好听，不需要露脸，厅播，纯绿色直播(不涉及任何违法违 规的 内容)岗位介绍:不露脸的聊天主播，会聊天即可。接受新人小白，免费培训，有很大晋升空间。 岗位要求:1.年龄:33岁以下 2.普通话标准，善于沟通 3.快手、抖音APP进行实名认证 4.在家即可 不需要来公司 5.每天直播四个小时，分两场(全天24小时自选) 注:1.免费培训"}'

In [13]:
import os
from openai import OpenAI

client = OpenAI(
    api_key="sk-4bc905f3de984dddba96572f21ccbb85",  # 如果您没有配置环境变量，请在此处用您的API Key进行替换
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"  # 百炼服务的base_url
)

response = client.embeddings.create(
    model="text-embedding-v4",
    input=str(jb_item),
    dimensions=1024,# 指定向量维度（仅 text-embedding-v3及 text-embedding-v4支持该参数）
    encoding_format="float"
)



In [20]:
json.loads(response.model_dump_json())['data'][0]['embedding']

[-0.0033276292961090803,
 -0.04796186462044716,
 0.07354571670293808,
 -0.05330492928624153,
 0.052959199994802475,
 -0.0012846927857026458,
 0.026463884860277176,
 0.07348285615444183,
 -0.04117302969098091,
 0.09422651678323746,
 0.0025104547385126352,
 0.00965680368244648,
 0.014496992342174053,
 0.00742135988548398,
 0.0384700670838356,
 0.0022727667819708586,
 -0.03472992032766342,
 -0.03614426031708717,
 -0.08819200098514557,
 -0.004930548835545778,
 -0.02519097924232483,
 0.019455041736364365,
 0.1039697527885437,
 0.03743288293480873,
 0.005975589156150818,
 0.022252293303608894,
 0.0017600683495402336,
 0.03435276448726654,
 0.014237696304917336,
 -0.03450991213321686,
 -0.008156031370162964,
 -0.006851695012301207,
 -0.012799782678484917,
 0.0004193912900518626,
 0.032074104994535446,
 0.021325115114450455,
 0.029103988781571388,
 -0.030172601342201233,
 -0.05591360107064247,
 0.015974191948771477,
 -0.005857727490365505,
 0.004121231380850077,
 -0.05126199126243591,
 -0.0275

In [19]:
from pymilvus import MilvusClient

# Authentication enabled with a cluster user
client = MilvusClient(
    uri = "https://in03-d2db10d0de909d0.serverless.aws-eu-central-1.cloud.zilliz.com",
    token = "607621de9dede7293c519ccc9f54390ae32c645a8a0b146aac648714263075083297cbd41843126dd8a1ea974821478e5b9607b9"
)

In [23]:
data = [{"id":str(jb_item.id), "embedding":json.loads(response.model_dump_json())['data'][0]['embedding']}]

In [25]:
client.insert(collection_name="intelli_job_job_items",
              data = data)

{'insert_count': 1, 'ids': ['51a7823a-6aa7-373a-b060-665e3461c04d'], 'cost': 2}